# Student Management System

## Database Setup

In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("Database created in memory.")

In [ ]:
# Create a students table
cursor.execute('''
    CREATE TABLE students (
        student_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        math_marks INTEGER,
        science_marks INTEGER,
        english_marks INTEGER
    )
''')

# Insert sample data
students_data = [
    (1, 'Alice Smith', 85, 90, 78),
    (2, 'Bob Johnson', 70, 65, 80),
    (3, 'Charlie Brown', 92, 88, 95),
    (4, 'Diana Prince', 60, 75, 70),
    (5, 'Eve Adams', 95, 92, 98)
]

cursor.executemany("INSERT INTO students (student_id, name, math_marks, science_marks, english_marks) VALUES (?, ?, ?, ?, ?)", students_data)
conn.commit()

print("Students table created and populated with sample data.")

## Calculate Average Marks

In [ ]:
# Calculate average marks for each student
query_avg_marks = '''
    SELECT
        student_id,
        name,
        (math_marks + science_marks + english_marks) / 3.0 AS average_marks
    FROM
        students
    ORDER BY
        average_marks DESC
'''

df_avg_marks = pd.read_sql_query(query_avg_marks, conn)
display(df_avg_marks)

## Display Top Performers

In [ ]:
# Display top 3 performers based on average marks
query_top_performers = '''
    SELECT
        student_id,
        name,
        (math_marks + science_marks + english_marks) / 3.0 AS average_marks
    FROM
        students
    ORDER BY
        average_marks DESC
    LIMIT 3
'''

df_top_performers = pd.read_sql_query(query_top_performers, conn)
display(df_top_performers)

## Perform SQL Queries

## Joins and OOP Concepts

In [ ]:
# Create a courses table
cursor.execute('''
    CREATE TABLE courses (
        course_id INTEGER PRIMARY KEY,
        course_name TEXT NOT NULL,
        credits INTEGER
    )
''')

# Create an enrollments table (junction table for many-to-many relationship)
cursor.execute('''
    CREATE TABLE enrollments (
        enrollment_id INTEGER PRIMARY KEY,
        student_id INTEGER,
        course_id INTEGER,
        grade TEXT,
        FOREIGN KEY (student_id) REFERENCES students(student_id),
        FOREIGN KEY (course_id) REFERENCES courses(course_id)
    )
''')

print("Courses and Enrollments tables created.")

In [ ]:
# Insert sample data into courses
courses_data = [
    (101, 'Mathematics I', 3),
    (102, 'Physics I', 4),
    (103, 'English Literature', 3),
    (104, 'Computer Science', 5)
]
cursor.executemany("INSERT INTO courses (course_id, course_name, credits) VALUES (?, ?, ?)", courses_data)

# Insert sample data into enrollments
enrollments_data = [
    (1, 1, 101, 'A'),
    (2, 1, 103, 'B'),
    (3, 2, 101, 'C'),
    (4, 3, 102, 'A'),
    (5, 3, 104, 'A'),
    (6, 4, 103, 'D'),
    (7, 5, 101, 'B+'),
    (8, 5, 102, 'A-')
]
cursor.executemany("INSERT INTO enrollments (enrollment_id, student_id, course_id, grade) VALUES (?, ?, ?, ?)", enrollments_data)

conn.commit()

print("Courses and Enrollments tables populated with sample data.")

### Example: SQL JOIN Query

In [ ]:
# Display students and their enrolled courses
print("\nStudents and their enrolled courses:")
query_student_courses = '''
    SELECT
        s.name AS student_name,
        c.course_name,
        e.grade
    FROM
        students s
    JOIN
        enrollments e ON s.student_id = e.student_id
    JOIN
        courses c ON e.course_id = c.course_id
    ORDER BY
        s.name, c.course_name
'''
df_student_courses = pd.read_sql_query(query_student_courses, conn)
display(df_student_courses)

### Refactoring with OOP Concepts

In [ ]:
class StudentManager:
    def __init__(self, db_connection):
        self.conn = db_connection
        self.cursor = self.conn.cursor()

    def get_all_students(self):
        query = "SELECT * FROM students"
        return pd.read_sql_query(query, self.conn)

    def get_student_by_name(self, name):
        query = f"SELECT * FROM students WHERE name = '{name}'"
        return pd.read_sql_query(query, self.conn)

    def get_students_with_math_above(self, score):
        query = f"SELECT name, math_marks FROM students WHERE math_marks > {score}"
        return pd.read_sql_query(query, self.conn)

    def get_average_marks(self):
        query = '''
            SELECT
                student_id,
                name,
                (math_marks + science_marks + english_marks) / 3.0 AS average_marks
            FROM
                students
            ORDER BY
                average_marks DESC
        '''
        return pd.read_sql_query(query, self.conn)

    def get_top_performers(self, limit=3):
        query = f'''
            SELECT
                student_id,
                name,
                (math_marks + science_marks + english_marks) / 3.0 AS average_marks
            FROM
                students
            ORDER BY
                average_marks DESC
            LIMIT {limit}
        '''
        return pd.read_sql_query(query, self.conn)

class CourseManager:
    def __init__(self, db_connection):
        self.conn = db_connection
        self.cursor = self.conn.cursor()

    def get_all_courses(self):
        query = "SELECT * FROM courses"
        return pd.read_sql_query(query, self.conn)

class EnrollmentManager:
    def __init__(self, db_connection):
        self.conn = db_connection
        self.cursor = self.conn.cursor()

    def get_student_enrollments(self):
        query = '''
            SELECT
                s.name AS student_name,
                c.course_name,
                e.grade
            FROM
                students s
            JOIN
                enrollments e ON s.student_id = e.student_id
            JOIN
                courses c ON e.course_id = c.course_id
            ORDER BY
                s.name, c.course_name
        '''
        return pd.read_sql_query(query, self.conn)

print("OOP classes for database interaction defined.")

### Demonstrating OOP Usage

In [ ]:
# Instantiate managers
student_mgr = StudentManager(conn)
course_mgr = CourseManager(conn)
enrollment_mgr = EnrollmentManager(conn)

# Using OOP to get all students
print("\nAll Students (via OOP):")
display(student_mgr.get_all_students())

# Using OOP to get student enrollments (demonstrating join via OOP method)
print("\nStudents and their enrolled courses (via OOP join method):")
display(enrollment_mgr.get_student_enrollments())

In [ ]:
# Example 1: Select all students
print("\nAll Students:")
df_all_students = pd.read_sql_query("SELECT * FROM students", conn)
display(df_all_students)

# Example 2: Find a student by name
print("\nStudent 'Alice Smith':")
df_alice = pd.read_sql_query("SELECT * FROM students WHERE name = 'Alice Smith'", conn)
display(df_alice)

# Example 3: Students with Math marks above 80
print("\nStudents with Math marks above 80:")
df_math_high = pd.read_sql_query("SELECT name, math_marks FROM students WHERE math_marks > 80", conn)
display(df_math_high)